# Fase 4 y 5 — Modeling + Evaluation (CRISP-DM)
## Sistema Inteligente de Recomendación de Películas — MovieLens 25M

Notebook `03_ML_Baseline_AutoML.ipynb` — **CRISP-DM Fase 4 (Modeling)** + **CRISP-DM Fase 5 (Evaluation)**.

Continuamos desde la Fase 3 (`02_Data_Sampling_and_Cleaning.ipynb`) que dejó listos dos artefactos:
- **Muestra principal al 60 %** para Baseline, SVD, NMF y AutoML.
- **Muestra al 10 %** reservada para KNN como benchmark metodológico.
- **`timestamp` preservado** para evaluación temporal por usuario.
- **Catálogo sin filtro global de cold-start**, de modo que el long tail se mida y no se elimine en preparación.

Este notebook cubre:
- **Modeling (Fase 4)**: selección de técnicas, entrenamiento y benchmark de modelos.
- **Evaluation (Fase 5)**: métricas unificadas, análisis temporal warm-start/cold-start, costo de modelado e interpretación de clústeres.


## Entregables de este notebook (especificaciones del proyecto)

| # | Entregable | Implementación en este notebook |
|---|---|---|
| 1 | **Baseline de recomendación** basado en similitud o popularidad. | Sección 3 — **Popularidad Bayesiana Ponderada** (fórmula IMDb) con *shrinkage* hacia la media global. |
| 2 | **Al menos 3 enfoques comparables** para predicción de rating/ranking. | Secciones 4-6 — **KNN-Baseline (10 % benchmark)**, **SVD (matrix factorization)**, **NMF (non-negative matrix factorization)**. |
| 3 | **AutoML como benchmark** (desempeño + costo). | Sección 7 — Auto-Surprise (TPE) si está disponible; fallback a **GridSearchCV multi-algoritmo** sobre la muestra principal. |
| 4 | **Clustering sobre usuarios o películas** + interpretación. | Sección 10 — **KMeans** sobre embeddings latentes de SVD (usuarios **y** películas), con perfiles de género, rating y ejemplos representativos. |
| 5 | **Métricas comparables**: RMSE, MAE (rating) + Precision@K, Recall@K, NDCG@K (ranking). | Secciones 8-9 — implementación propia de Top-N sobre el test temporal warm-start. |

### Decisiones metodológicas clave
1. **Biblioteca**: `scikit-surprise` — estándar académico para CF, expone `.pu`/`.qi` (factores latentes) necesarios para el clustering.
2. **Dataset principal**: muestra estratificada al `60 %` por *tier* de actividad del usuario, sin filtro global de popularidad. Motivo: preservar catálogo y seguir siendo factible en hardware medio.
3. **Evaluación**: `leave-last-1-out` temporal por usuario. Se evita leakage y se separa explícitamente el test `warm` del `cold-start` real.
4. **KNN**: se conserva sólo como benchmark metodológico sobre una muestra estratificada al `10 %`, porque su costo crece mal con el tamaño del catálogo.
5. **Umbral de relevante (Top-N)**: 4.0 (no 3.5) — se ajusta al sesgo positivo del dataset para no inflar `Recall`.
6. **AutoML sub-sample**: 20 % del train principal para mantener la búsqueda factible.


In [ ]:
# ============== FORZAR VENV LOCAL ==============
import sys
from pathlib import Path
_venv_path = str((Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()) / "venv" / "lib" / "python3.12" / "site-packages")
if _venv_path not in sys.path:
    sys.path.insert(0, _venv_path)
    print(f"Ruta del venv inyectada en sys.path: {_venv_path}")

import os, gc, sys, time, random, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

try:
    from surprise import Dataset, Reader, accuracy
    from surprise import SVD, NMF, KNNBaseline, BaselineOnly
    from surprise.model_selection import GridSearchCV as SurpriseGridSearch
    from surprise.prediction_algorithms.predictions import Prediction
    SURPRISE_OK = True
except ImportError as e:
    SURPRISE_OK = False
    print()
    print('ERROR IMPORTANDO SURPRISE:', e)
    print('Verifica la versión de numpy o reinstala scikit-surprise.')
    print()

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

try:
    from auto_surprise.engine import Engine as AutoSurpriseEngine  # noqa: F401
    AUTO_SURPRISE_AVAILABLE = True
except Exception as _e:
    AUTO_SURPRISE_AVAILABLE = False
    _autosurp_err = str(_e)

print(f'scikit-surprise : OK')
print(f'auto-surprise   : {"disponible" if AUTO_SURPRISE_AVAILABLE else "no instalado (se usará fallback GridSearchCV)"}')


In [ ]:
# ============== REPRODUCIBILIDAD + RUTAS ==============
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

RELEVANT_THRESHOLD = 4.0
K_VALUES           = (5, 10)
AUTOML_FRACTION    = 0.20
AUTOML_TIME_BUDGET = 600
K_USERS_CLUSTERS   = 6
K_ITEMS_CLUSTERS   = 6

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_RAW_DIR = ROOT / 'data' / 'ml-25m'
DATA_INT_DIR = ROOT / 'data' / 'intermediate'
MODELS_DIR   = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_INT_DIR.mkdir(parents=True, exist_ok=True)

vm = psutil.virtual_memory()
print(f'RAM total      : {vm.total / 1024**3:5.1f} GB')
print(f'RAM disponible : {vm.available / 1024**3:5.1f} GB')
print(f'CPUs           : {psutil.cpu_count(logical=False)} físicos / {psutil.cpu_count(logical=True)} lógicos')
print(f'Raíz proyecto  : {ROOT}')
if vm.available / 1024**3 < 4:
    print('ADVERTENCIA: RAM libre < 4 GB. Considera reiniciar el kernel antes de continuar.')


## 1. Carga de datos
Leemos dos datasets preparados en `02_Data_Sampling_and_Cleaning.ipynb`:

- `ratings_prepared_60pct.parquet` + `movies_prepared_60pct.parquet` para Baseline, SVD, NMF y AutoML.
- `ratings_knn_10pct.parquet` + `movies_knn_10pct.parquet` para el benchmark KNN.

No se reconstruye automáticamente el sample viejo de `5 % + cold-start`; si faltan los parquets, primero hay que ejecutar el notebook 02.


In [ ]:
# ============== CARGA DE DATASETS PREPARADOS ==============
RATINGS_PARQUET = DATA_INT_DIR / 'ratings_prepared_60pct.parquet'
MOVIES_PARQUET  = DATA_INT_DIR / 'movies_prepared_60pct.parquet'
KNN_RATINGS_PARQUET = DATA_INT_DIR / 'ratings_knn_10pct.parquet'
KNN_MOVIES_PARQUET  = DATA_INT_DIR / 'movies_knn_10pct.parquet'

required = [RATINGS_PARQUET, MOVIES_PARQUET, KNN_RATINGS_PARQUET, KNN_MOVIES_PARQUET]
missing = [p.name for p in required if not p.exists()]
assert not missing, f'Faltan artefactos preparados: {missing}. Ejecuta antes el notebook 02.'

ratings = pd.read_parquet(RATINGS_PARQUET).astype({
    'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'
})
movies = pd.read_parquet(MOVIES_PARQUET)
ratings_knn = pd.read_parquet(KNN_RATINGS_PARQUET).astype({
    'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'
})
movies_knn = pd.read_parquet(KNN_MOVIES_PARQUET)

print(f'[MAIN 60%] Ratings         : {len(ratings):>10,}')
print(f'[MAIN 60%] Usuarios únicos : {ratings["userId"].nunique():>10,}')
print(f'[MAIN 60%] Películas únic. : {ratings["movieId"].nunique():>10,}')
print(f'[MAIN 60%] Media rating    : {ratings["rating"].mean():.3f}')
print('-' * 60)
print(f'[KNN 10%] Ratings          : {len(ratings_knn):>10,}')
print(f'[KNN 10%] Usuarios únicos  : {ratings_knn["userId"].nunique():>10,}')
print(f'[KNN 10%] Películas únic.  : {ratings_knn["movieId"].nunique():>10,}')
print(f'[KNN 10%] Media rating     : {ratings_knn["rating"].mean():.3f}')


In [ ]:
# ============== SPLIT TEMPORAL POR USUARIO ==============
reader = Reader(rating_scale=(0.5, 5.0))

def temporal_leave_last_one_out(df, label):
    ordered = df.sort_values(['userId', 'timestamp', 'movieId']).reset_index(drop=True)
    ordered['event_rank'] = ordered.groupby('userId').cumcount()

    test_df = ordered.groupby('userId', group_keys=False).tail(1).copy()
    train_df = ordered.drop(test_df.index).copy()

    train_items = set(train_df['movieId'].unique().tolist())
    test_warm_df = test_df[test_df['movieId'].isin(train_items)].copy().reset_index(drop=True)
    test_cold_items_df = test_df[~test_df['movieId'].isin(train_items)].copy().reset_index(drop=True)

    assert len(train_df) + len(test_df) == len(ordered)
    assert (train_df.groupby('userId')['event_rank'].max() < test_df.groupby('userId')['event_rank'].min()).all()
    assert test_warm_df['movieId'].isin(train_items).all()
    assert not test_cold_items_df['movieId'].isin(train_items).any()

    train_df = train_df.drop(columns='event_rank').reset_index(drop=True)
    test_df = test_df.drop(columns='event_rank').reset_index(drop=True)
    test_warm_df = test_warm_df.drop(columns='event_rank', errors='ignore')
    test_cold_items_df = test_cold_items_df.drop(columns='event_rank', errors='ignore')

    trainset = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader).build_full_trainset()
    testset_warm = list(test_warm_df[['userId', 'movieId', 'rating']].itertuples(index=False, name=None))

    print(f'[{label}] Train ratings          : {len(train_df):,}')
    print(f'[{label}] Test temporal total    : {len(test_df):,}')
    print(f'[{label}] Test warm-start        : {len(test_warm_df):,}')
    print(f'[{label}] Test cold-start items  : {len(test_cold_items_df):,} ({100*len(test_cold_items_df)/len(test_df):.2f}%)')
    print(f'[{label}] Usuarios (trainset)    : {trainset.n_users:,}')
    print(f'[{label}] Items (trainset)       : {trainset.n_items:,}')
    print(f'[{label}] Media global (μ)       : {trainset.global_mean:.4f}')

    return train_df, test_df.reset_index(drop=True), test_warm_df, test_cold_items_df, trainset, testset_warm

(train_main_df, test_main_df, test_main_warm_df, test_main_cold_items_df, trainset, testset_warm) = temporal_leave_last_one_out(ratings, 'MAIN 60%')
(train_knn_df, test_knn_df, test_knn_warm_df, test_knn_cold_items_df, trainset_knn, testset_knn_warm) = temporal_leave_last_one_out(ratings_knn, 'KNN 10%')

GLOBAL_MEAN = trainset.global_mean


## 2. Baseline — Popularidad Bayesiana Ponderada

**Justificación.** El PDF exige un baseline explícito "basado en popularidad o similitud". Usamos la **fórmula de IMDb** (Weighted Rating) que combina la media por película con la media global vía *shrinkage*:

$$WR(i) = \frac{v_i}{v_i + m} \cdot R_i + \frac{m}{v_i + m} \cdot C$$

donde:
- $v_i$ = número de votos recibidos por la película $i$.
- $R_i$ = rating promedio de la película $i$.
- $C$ = media global de todos los ratings en el trainset.
- $m$ = umbral de votos para estabilizar (percentil 60 de conteo).

**Por qué este baseline y no un simple `rating_mean`:**
- Penaliza películas con pocos votos (regula el ruido de la *long tail* identificado en el EDA).
- Produce un ordenamiento consistente para recomendaciones Top-N (no sólo para RMSE).
- Es ampliamente usado en sistemas productivos (IMDb, Rotten Tomatoes) → ancla interpretable.


In [ ]:
# ============== BASELINE: BAYESIAN WEIGHTED RATING ==============
t0 = time.time()

train_ratings_df = train_main_df[['userId', 'movieId', 'rating']].copy()
movie_stats = train_ratings_df.groupby('movieId').agg(
    n_votes=('rating', 'count'),
    mean_rating=('rating', 'mean')
).reset_index()

C = train_ratings_df['rating'].mean()
m = movie_stats['n_votes'].quantile(0.60)

movie_stats['bayesian_score'] = (
    (movie_stats['n_votes'] / (movie_stats['n_votes'] + m)) * movie_stats['mean_rating']
    + (m / (movie_stats['n_votes'] + m)) * C
)
score_dict = dict(zip(movie_stats['movieId'], movie_stats['bayesian_score']))

baseline_preds = [
    Prediction(uid=u, iid=i, r_ui=r, est=float(score_dict.get(i, C)), details={})
    for (u, i, r) in testset_warm
]
baseline_time = time.time() - t0

baseline_rmse = accuracy.rmse(baseline_preds, verbose=False)
baseline_mae  = accuracy.mae(baseline_preds, verbose=False)

print(f'Baseline (Pop. Bayesiana)   RMSE = {baseline_rmse:.4f}   MAE = {baseline_mae:.4f}   Tiempo = {baseline_time:.1f}s')
print(f'Parámetros: C (global mean) = {C:.4f}   m (votos prior) = {m:.0f}')
print(f'Cobertura cold-start (fuera de métrica warm): {len(test_main_cold_items_df):,} interacciones')
print()

top_pop = movie_stats.merge(movies[['movieId', 'title', 'genres']], on='movieId', how='left')
top_pop = top_pop.sort_values('bayesian_score', ascending=False).head(15)
print('Top-15 películas ranqueadas por el baseline:')
print(top_pop[['title', 'n_votes', 'mean_rating', 'bayesian_score']].to_string(index=False))


## 3. Modelo clásico #1 — KNN-Baseline (item-based)

**Justificación.** KNN se mantiene como benchmark interpretable del filtrado colaborativo *memory-based*, pero ya no se entrena sobre la muestra principal:
- **Dataset dedicado al 10 %**: limita el costo de la matriz item-item y mantiene el experimento factible.
- **Item-based**: la matriz ítem-ítem es más estable en el tiempo que usuario-usuario.
- **Similitud `pearson_baseline`**: corrige sesgos por usuario/ítem antes de calcular similitudes.

Este modelo sigue siendo útil para comparar contra SVD/NMF, pero queda fuera del pipeline principal por escalabilidad.


In [ ]:
# ============== KNN-BASELINE (ITEM-BASED, 10%) ==============
t0 = time.time()
knn = KNNBaseline(
    k=40,
    sim_options={'name': 'pearson_baseline', 'user_based': False, 'min_support': 5},
    bsl_options={'method': 'als', 'n_epochs': 10, 'reg_u': 12, 'reg_i': 5},
    verbose=False,
)
knn.fit(trainset_knn)
knn_preds = knn.test(testset_knn_warm)
knn_time = time.time() - t0

knn_rmse = accuracy.rmse(knn_preds, verbose=False)
knn_mae  = accuracy.mae(knn_preds, verbose=False)
print(f'KNN-Baseline (item-based, 10%)   RMSE = {knn_rmse:.4f}   MAE = {knn_mae:.4f}   Tiempo = {knn_time:.1f}s')
print(f'KNN cold-start items fuera de warm-test: {len(test_knn_cold_items_df):,}')


## 4. Modelo clásico #2 — SVD (Matrix Factorization)

**Justificación.** SVD es el modelo principal porque:
- **Maneja sparsity extrema** aprendiendo factores latentes densos para cada usuario y cada ítem.
- **Los factores aprendidos (`.pu`, `.qi`)** se reutilizan luego para clustering.
- **Escala linealmente en el número de ratings por época**, mucho mejor que KNN cuando el catálogo crece.

Se entrena sobre la muestra principal al `60 %` y se evalúa con split temporal warm-start.


In [ ]:
# ============== SVD (MATRIX FACTORIZATION, 60%) ==============
t0 = time.time()
svd = SVD(
    n_factors=50, n_epochs=20,
    lr_all=0.005, reg_all=0.02,
    random_state=SEED,
)
svd.fit(trainset)
svd_preds = svd.test(testset_warm)
svd_time = time.time() - t0

svd_rmse = accuracy.rmse(svd_preds, verbose=False)
svd_mae  = accuracy.mae(svd_preds, verbose=False)
print(f'SVD (Matrix Factorization, 60%)  RMSE = {svd_rmse:.4f}   MAE = {svd_mae:.4f}   Tiempo = {svd_time:.1f}s')
print(f'Factores latentes: usuarios = {svd.pu.shape}   ítems = {svd.qi.shape}')


## 5. Modelo clásico #3 — NMF (Non-negative Matrix Factorization)

**Justificación.** NMF añade la restricción de **no negatividad** en los factores, lo que produce representaciones más interpretables y sirve como comparador matricial de SVD.

Se mantiene sobre la muestra principal al `60 %` para que la comparación principal sea entre modelos escalables bajo el mismo split temporal.


In [ ]:
# ============== NMF (60%) ==============
t0 = time.time()
nmf = NMF(
    n_factors=15, n_epochs=50,
    biased=False,
    random_state=SEED,
)
nmf.fit(trainset)
nmf_preds = nmf.test(testset_warm)
nmf_time = time.time() - t0

nmf_rmse = accuracy.rmse(nmf_preds, verbose=False)
nmf_mae  = accuracy.mae(nmf_preds, verbose=False)
print(f'NMF (60%)                     RMSE = {nmf_rmse:.4f}   MAE = {nmf_mae:.4f}   Tiempo = {nmf_time:.1f}s')


## 6. AutoML — Benchmark de búsqueda automática de hiperparámetros

**Justificación.** AutoML sigue siendo un benchmark obligatorio, pero ahora se limita a los modelos del pipeline principal (`SVD`, `NMF`, `BaselineOnly`) sobre una submuestra del train temporal del `60 %`.

Esto evita mezclar KNN en una búsqueda costosa que no forma parte del pipeline operativo principal.


In [ ]:
# ============== AUTOML BENCHMARK (SOBRE TRAIN 60%) ==============
automl_start = time.time()

mini_train_df = train_main_df.sample(frac=AUTOML_FRACTION, random_state=SEED)
mini_data = Dataset.load_from_df(mini_train_df[['userId', 'movieId', 'rating']], reader)
print(f'Sub-muestra AutoML: {len(mini_train_df):,} ratings ({AUTOML_FRACTION*100:.0f}% del train principal)')

automl_results = {}

if AUTO_SURPRISE_AVAILABLE:
    print()
    print('>> Ejecutando auto-surprise (optimización bayesiana TPE)...')
    engine = AutoSurpriseEngine(verbose=1, random_state=SEED)
    best_algo, best_params, best_score, tasks = engine.train(
        data=mini_data,
        target_metric='test_rmse',
        cpu_time_limit=AUTOML_TIME_BUDGET,
        max_evals=30,
    )
    automl_results['best_algo'] = str(best_algo)
    automl_results['best_params'] = dict(best_params)
    automl_results['best_rmse'] = float(best_score)
    automl_results['method'] = 'auto-surprise (TPE)'
else:
    print('>> auto-surprise no disponible. Ejecutando GridSearchCV multi-algoritmo (fallback)...')
    search_space = {
        'SVD': (SVD, {
            'n_factors': [30, 50, 100],
            'n_epochs':  [15, 25],
            'lr_all':    [0.005, 0.01],
            'reg_all':   [0.02, 0.05],
        }),
        'NMF': (NMF, {
            'n_factors': [10, 15, 25],
            'n_epochs':  [40, 60],
        }),
        'BaselineOnly': (BaselineOnly, {
            'bsl_options': {'method': ['als', 'sgd'], 'n_epochs': [10, 20]},
        }),
    }

    per_algo = {}
    for name, (AlgoCls, grid) in search_space.items():
        t0 = time.time()
        print(f'   • {name:<15} — explorando hiperparámetros...')
        gs = SurpriseGridSearch(AlgoCls, grid, measures=['rmse', 'mae'], cv=2, n_jobs=1, joblib_verbose=0)
        gs.fit(mini_data)
        per_algo[name] = {
            'rmse':   float(gs.best_score['rmse']),
            'mae':    float(gs.best_score['mae']),
            'params': dict(gs.best_params['rmse']),
            'time':   time.time() - t0,
        }
        print(f'     └─ best RMSE = {per_algo[name]["rmse"]:.4f}   best params = {per_algo[name]["params"]}   ({per_algo[name]["time"]:.1f}s)')

    best_algo_name = min(per_algo, key=lambda n: per_algo[n]['rmse'])
    automl_results = {
        'best_algo': best_algo_name,
        'best_params': per_algo[best_algo_name]['params'],
        'best_rmse': per_algo[best_algo_name]['rmse'],
        'best_mae':  per_algo[best_algo_name]['mae'],
        'per_algo':  per_algo,
        'method':    'GridSearchCV (fallback)',
    }

automl_total_time = time.time() - automl_start
print()
print('AutoML resumen:')
print(f'   método          : {automl_results["method"]}')
print(f'   mejor algoritmo : {automl_results["best_algo"]}')
print(f'   mejor RMSE (CV) : {automl_results["best_rmse"]:.4f}')
print(f'   mejores params  : {automl_results["best_params"]}')
print(f'   tiempo total    : {automl_total_time:.1f}s')


In [ ]:
# ============== ENTRENAR EL MEJOR MODELO AUTOML SOBRE TRAIN 60% ==============
best_name = automl_results['best_algo']
best_params = automl_results['best_params']

algo_map = {'SVD': SVD, 'NMF': NMF, 'BaselineOnly': BaselineOnly}
AlgoCls = algo_map.get(best_name, SVD)

try:
    automl_model = AlgoCls(**best_params, random_state=SEED)
except TypeError:
    automl_model = AlgoCls(**best_params)

t0 = time.time()
automl_model.fit(trainset)
automl_preds = automl_model.test(testset_warm)
automl_train_time = time.time() - t0

automl_rmse = accuracy.rmse(automl_preds, verbose=False)
automl_mae  = accuracy.mae(automl_preds, verbose=False)
print(f'AutoML winner ({best_name}, 60%) refiteado   RMSE = {automl_rmse:.4f}   MAE = {automl_mae:.4f}   Tiempo fit = {automl_train_time:.1f}s')


## 7. Métricas Top-N — Precision@K, Recall@K, NDCG@K

Las métricas RMSE/MAE evalúan **predicción de rating** pero no **ranking**. Dado que el objetivo de negocio es recomendar (Top-N), también calculamos métricas de ranking sobre el **test temporal warm-start**.

El conjunto `cold-start` se reporta aparte como cobertura del problema real, porque un CF puro no puede puntuar ítems que nunca aparecieron en train.


In [ ]:
# ============== IMPLEMENTACIÓN DE MÉTRICAS TOP-N ==============
def precision_recall_at_k(predictions, k=10, threshold=RELEVANT_THRESHOLD):
    """Calcula Precision@K y Recall@K promediados sobre usuarios."""
    user_est_true = defaultdict(list)
    for p in predictions:
        user_est_true[p.uid].append((p.est, p.r_ui))
    precisions, recalls = {}, {}
    for uid, ur in user_est_true.items():
        ur.sort(key=lambda x: x[0], reverse=True)
        n_rel        = sum((true_r >= threshold) for _, true_r in ur)
        n_rec_k      = sum((est >= threshold) for est, _ in ur[:k])
        n_rel_and_k  = sum(((true_r >= threshold) and (est >= threshold))
                           for est, true_r in ur[:k])
        precisions[uid] = n_rel_and_k / n_rec_k if n_rec_k else 0.0
        recalls[uid]    = n_rel_and_k / n_rel   if n_rel   else 0.0
    return (np.mean(list(precisions.values())),
            np.mean(list(recalls.values())))

def ndcg_at_k(predictions, k=10):
    """NDCG@K promedio sobre usuarios (relevancia = rating real)."""
    user_est_true = defaultdict(list)
    for p in predictions:
        user_est_true[p.uid].append((p.est, p.r_ui))
    def _dcg(rels):
        return sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rels))
    scores = []
    for uid, ur in user_est_true.items():
        if len(ur) < 2:
            continue
        by_pred = sorted(ur, key=lambda x: x[0], reverse=True)[:k]
        by_true = sorted(ur, key=lambda x: x[1], reverse=True)[:k]
        d    = _dcg([r for _, r in by_pred])
        idcg = _dcg([r for _, r in by_true])
        scores.append(d / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores)) if scores else 0.0

print('Funciones Top-N definidas.')


## 8. Evaluación unificada de los modelos


In [ ]:
# ============== EVALUAR TODOS LOS MODELOS ==============
models_preds = {
    'Baseline (Pop. Bayesiana)': baseline_preds,
    'SVD':                       svd_preds,
    'NMF':                       nmf_preds,
    f'AutoML → {best_name}':     automl_preds,
    'KNN-Baseline (item-based)': knn_preds,
}
model_times = {
    'Baseline (Pop. Bayesiana)': baseline_time,
    'SVD':                       svd_time,
    'NMF':                       nmf_time,
    f'AutoML → {best_name}':     automl_train_time + automl_total_time,
    'KNN-Baseline (item-based)': knn_time,
}
model_dataset = {
    'Baseline (Pop. Bayesiana)': 'MAIN 60%',
    'SVD':                       'MAIN 60%',
    'NMF':                       'MAIN 60%',
    f'AutoML → {best_name}':     'MAIN 60%',
    'KNN-Baseline (item-based)': 'KNN 10%',
}
model_cold = {
    'Baseline (Pop. Bayesiana)': len(test_main_cold_items_df),
    'SVD':                       len(test_main_cold_items_df),
    'NMF':                       len(test_main_cold_items_df),
    f'AutoML → {best_name}':     len(test_main_cold_items_df),
    'KNN-Baseline (item-based)': len(test_knn_cold_items_df),
}

rows = []
for name, preds in models_preds.items():
    p5,  r5  = precision_recall_at_k(preds, k=5,  threshold=RELEVANT_THRESHOLD)
    p10, r10 = precision_recall_at_k(preds, k=10, threshold=RELEVANT_THRESHOLD)
    ndcg10   = ndcg_at_k(preds, k=10)
    rmse = accuracy.rmse(preds, verbose=False)
    mae  = accuracy.mae(preds, verbose=False)
    rows.append({
        'Modelo':      name,
        'Dataset':     model_dataset[name],
        'Test':        'Temporal warm-start',
        'Cold items':  model_cold[name],
        'RMSE':        rmse,
        'MAE':         mae,
        'P@5':         p5,
        'R@5':         r5,
        'P@10':        p10,
        'R@10':        r10,
        'NDCG@10':     ndcg10,
        'Tiempo (s)':  model_times[name],
    })

comp_df = pd.DataFrame(rows)
comp_df = comp_df.sort_values(['Dataset', 'RMSE']).reset_index(drop=True)
print(comp_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


In [ ]:
# ============== VISUALIZACIÓN COMPARATIVA ==============
plot_df = comp_df[comp_df['Dataset'] == 'MAIN 60%'].reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = sns.color_palette('viridis', n_colors=len(plot_df))
axes[0].barh(plot_df['Modelo'], plot_df['RMSE'], color=colors)
axes[0].set_title('RMSE por modelo principal (60%)')
axes[0].set_xlabel('RMSE')
axes[0].invert_yaxis()
for i, v in enumerate(plot_df['RMSE']):
    axes[0].text(v, i, f' {v:.4f}', va='center', fontsize=9)

axes[1].barh(plot_df['Modelo'], plot_df['NDCG@10'], color=colors)
axes[1].set_title('NDCG@10 (60%)')
axes[1].set_xlabel('NDCG@10')
axes[1].invert_yaxis()
for i, v in enumerate(plot_df['NDCG@10']):
    axes[1].text(v, i, f' {v:.4f}', va='center', fontsize=9)

axes[2].barh(comp_df['Modelo'], comp_df['Tiempo (s)'], color=sns.color_palette('magma', n_colors=len(comp_df)))
axes[2].set_title('Costo de entrenamiento (todos los modelos)')
axes[2].set_xlabel('Segundos (log)')
axes[2].set_xscale('log')
axes[2].invert_yaxis()
for i, v in enumerate(comp_df['Tiempo (s)']):
    axes[2].text(v, i, f' {v:.1f}s', va='center', fontsize=9)

plt.tight_layout(); plt.show()

comp_df.to_csv(DATA_INT_DIR / 'model_comparison.csv', index=False)
print(f'Tabla exportada a: {DATA_INT_DIR / "model_comparison.csv"}')


**Lectura de la tabla comparativa.**
- El bloque principal (`60 %`) compara modelos escalables bajo el mismo split temporal warm-start.
- **SVD** sigue siendo el candidato principal por equilibrio entre calidad, tiempo y reutilización de embeddings.
- **KNN** se mantiene como benchmark metodológico sobre `10 %`: sirve para comparar enfoques *memory-based* vs *model-based*, pero ya no condiciona el pipeline principal.
- La columna `Cold items` muestra cuántas últimas interacciones quedaron fuera del warm-start por ser ítems nunca vistos en train.


## 9. Clustering no supervisado sobre embeddings latentes

**Justificación.** SVD aprende, para cada usuario y cada película, un vector denso de 50 dimensiones (los factores latentes). Esos vectores son un **espacio semántico** donde la proximidad euclidiana equivale a "gustos similares" (para usuarios) o "películas parecidas" (para ítems).

Aplicamos **KMeans** sobre ambos espacios:
- Sobre los **vectores de usuario** (`svd.pu`) → segmentos de audiencia ("fans de acción", "cinéfilos de drama", etc.).
- Sobre los **vectores de película** (`svd.qi`) → *nichos* del catálogo ("blockbusters familiares", "cine de autor", "series de acción").

Este es el **componente no supervisado obligatorio** del proyecto. Se interpretan los clústeres cruzando con `movies.genres` y con las medias de rating.


In [ ]:
# ============== EXTRAER EMBEDDINGS DEL SVD ==============
user_factors = svd.pu            # shape (n_users_train, n_factors)
item_factors = svd.qi            # shape (n_items_train, n_factors)

# Mapeo inner → raw IDs (Surprise usa IDs internos consecutivos)
inner_to_raw_u = np.array([trainset.to_raw_uid(i) for i in range(trainset.n_users)])
inner_to_raw_i = np.array([trainset.to_raw_iid(i) for i in range(trainset.n_items)])

print(f'User embeddings  : {user_factors.shape}')
print(f'Item embeddings  : {item_factors.shape}')


In [ ]:
# ============== ELBOW + SILHOUETTE PARA ELEGIR K ==============
k_range = list(range(2, 11))
inertias_u, silh_u = [], []
inertias_i, silh_i = [], []

rng = np.random.RandomState(SEED)
u_idx_sil = rng.choice(len(user_factors), size=min(4000, len(user_factors)), replace=False)
i_idx_sil = rng.choice(len(item_factors), size=min(4000, len(item_factors)), replace=False)

for k in k_range:
    km_u = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(user_factors)
    km_i = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(item_factors)
    inertias_u.append(km_u.inertia_)
    inertias_i.append(km_i.inertia_)
    silh_u.append(silhouette_score(user_factors[u_idx_sil], km_u.labels_[u_idx_sil]))
    silh_i.append(silhouette_score(item_factors[i_idx_sil], km_i.labels_[i_idx_sil]))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(k_range, inertias_u, 'o-', color='#3182bd'); axes[0, 0].set_title('Elbow — Usuarios'); axes[0, 0].set_xlabel('k'); axes[0, 0].set_ylabel('Inertia')
axes[0, 1].plot(k_range, silh_u,     'o-', color='#e6550d'); axes[0, 1].set_title('Silhouette — Usuarios'); axes[0, 1].set_xlabel('k'); axes[0, 1].set_ylabel('score')
axes[1, 0].plot(k_range, inertias_i, 'o-', color='#31a354'); axes[1, 0].set_title('Elbow — Películas'); axes[1, 0].set_xlabel('k'); axes[1, 0].set_ylabel('Inertia')
axes[1, 1].plot(k_range, silh_i,     'o-', color='#756bb1'); axes[1, 1].set_title('Silhouette — Películas'); axes[1, 1].set_xlabel('k'); axes[1, 1].set_ylabel('score')
plt.tight_layout(); plt.show()

print(f'Usando K_USERS={K_USERS_CLUSTERS} y K_ITEMS={K_ITEMS_CLUSTERS} (configurables arriba en la sección de hiperparámetros).')


In [ ]:
# ============== KMEANS FINAL Y PERFILADO DE CLÚSTERES DE USUARIOS ==============
km_u_final = KMeans(n_clusters=K_USERS_CLUSTERS, random_state=SEED, n_init=10)
user_clusters = km_u_final.fit_predict(user_factors)

user_cluster_df = pd.DataFrame({
    'userId':  inner_to_raw_u.astype(np.int32),
    'cluster': user_clusters,
})

# Combinar con ratings y géneros para interpretar
tr_ratings = train_ratings_df.merge(user_cluster_df, on='userId', how='inner')
tr_ratings = tr_ratings.merge(movies[['movieId', 'genres']], on='movieId', how='left')

profiles = tr_ratings.groupby('cluster').agg(
    n_users=('userId', 'nunique'),
    n_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean'),
    std_rating=('rating', 'std'),
)
profiles['ratings_per_user'] = (profiles['n_ratings'] / profiles['n_users']).round(1)
print('Perfiles agregados por clúster de USUARIOS:')
print(profiles.round(3))

# Géneros top por clúster
def top_genres(df, topn=3):
    exploded = df.assign(genre=df['genres'].str.split('|')).explode('genre')
    return (exploded.groupby('genre')['rating']
            .agg(['count', 'mean'])
            .sort_values('count', ascending=False)
            .head(topn))

print('\nGéneros más consumidos por clúster (con su rating promedio):')
for c in sorted(tr_ratings['cluster'].unique()):
    top = top_genres(tr_ratings[tr_ratings['cluster'] == c])
    print(f'\n  Cluster {c} ({profiles.loc[c, "n_users"]} usuarios, rating_mean={profiles.loc[c, "avg_rating"]:.2f}):')
    for g, row in top.iterrows():
        print(f'    · {g:<20} n_votos={int(row["count"]):>7,}   rating_mean={row["mean"]:.2f}')


In [ ]:
# ============== KMEANS FINAL Y PERFILADO DE CLÚSTERES DE PELÍCULAS ==============
km_i_final = KMeans(n_clusters=K_ITEMS_CLUSTERS, random_state=SEED, n_init=10)
item_clusters = km_i_final.fit_predict(item_factors)

item_cluster_df = pd.DataFrame({
    'movieId': inner_to_raw_i.astype(np.int32),
    'cluster': item_clusters,
}).merge(movies, on='movieId', how='left')

# Popularidad y rating medio por película (del trainset, coherente con los embeddings)
pop_by_movie = train_ratings_df.groupby('movieId').agg(
    n_ratings=('rating', 'count'),
    rating_mean=('rating', 'mean'),
).reset_index()
item_cluster_df = item_cluster_df.merge(pop_by_movie, on='movieId', how='left')

# Perfiles
item_profiles = item_cluster_df.groupby('cluster').agg(
    n_movies=('movieId', 'count'),
    avg_rating=('rating_mean', 'mean'),
    avg_popularity=('n_ratings', 'mean'),
    max_popularity=('n_ratings', 'max'),
).round(2)
print('Perfiles agregados por clúster de PELÍCULAS:')
print(item_profiles)

# Géneros dominantes y ejemplos representativos
print('\nEjemplos representativos por clúster (top-3 más populares de cada uno):')
for c in sorted(item_cluster_df['cluster'].unique()):
    sub = item_cluster_df[item_cluster_df['cluster'] == c].dropna(subset=['n_ratings']).sort_values('n_ratings', ascending=False)
    # Géneros dominantes
    g_exp = sub.assign(g=sub['genres'].str.split('|')).explode('g')
    top_g = g_exp['g'].value_counts().head(3).index.tolist()
    print(f'\n  Cluster {c} ({item_profiles.loc[c, "n_movies"]} películas, rating_mean={item_profiles.loc[c, "avg_rating"]:.2f}, '
          f'popularidad media={item_profiles.loc[c, "avg_popularity"]:.0f})')
    print(f'    géneros dominantes: {", ".join(top_g)}')
    for _, row in sub.head(3).iterrows():
        print(f'    ★ {row["title"]}  ({row["genres"]})  n={int(row["n_ratings"])}, avg={row["rating_mean"]:.2f}')


In [ ]:
# ============== VISUALIZACIÓN PCA 2D DE CLÚSTERES ==============
pca_u = PCA(n_components=2, random_state=SEED).fit_transform(user_factors)
pca_i = PCA(n_components=2, random_state=SEED).fit_transform(item_factors)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(pca_u[:, 0], pca_u[:, 1], c=user_clusters, cmap='tab10', s=8, alpha=0.6)
axes[0].set_title(f'Embeddings de USUARIOS — PCA 2D  (k={K_USERS_CLUSTERS})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0], label='cluster')

sc2 = axes[1].scatter(pca_i[:, 0], pca_i[:, 1], c=item_clusters, cmap='tab10', s=8, alpha=0.6)
axes[1].set_title(f'Embeddings de PELÍCULAS — PCA 2D  (k={K_ITEMS_CLUSTERS})')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc2, ax=axes[1], label='cluster')

plt.tight_layout(); plt.show()


**Interpretación de clústeres (guía).**

Los clústeres de **usuarios** suelen segmentarse en perfiles como:
- *Críticos exigentes*: rating medio bajo (<3.5), varianza alta, consumen géneros nicho (Documentary, War).
- *Entusiastas*: rating medio alto (>4.0), pocos votos, mayormente blockbusters.
- *Cinéfilos balanceados*: rating medio ≈ global, muchísimos votos, todos los géneros.
- *Casuales*: pocos ratings, concentrados en Drama/Comedy mainstream.

Los clústeres de **películas** suelen separarse en:
- *Blockbusters universales* (alto rating + mucha popularidad).
- *Cine de nicho bien calificado* (alto rating + baja popularidad) → oro para **recomendaciones de descubrimiento**.
- *Contenido mainstream de rating medio* (volumen + rating ≈ 3.5).
- *Cola larga ruidosa* (pocos votos, dispersión alta).

Estos segmentos habilitan:
- **Recomendaciones segmentadas**: seleccionar candidatos dentro del mismo clúster del usuario activo.
- **Diversificación**: forzar que el Top-N cubra ≥ 2 clústeres distintos de películas.
- **Cold-start de usuario**: asignar al clúster más cercano según sus primeros 5-10 ratings.


## 10. Conclusiones y siguiente paso

| Hallazgo | Decisión para Fase 4 (Deep Learning) |
|---|---|
| SVD domina el bloque principal (`60 %`) con costo moderado y embeddings reutilizables. | Próximo modelo: **Neural Collaborative Filtering** sobre el mismo split temporal. |
| KNN aporta comparación metodológica, pero sólo es factible en `10 %`. | Mantenerlo como benchmark, no como modelo operativo principal. |
| El winner de AutoML confirma si el tuneo clásico mejora materialmente al SVD base. | No invertir más ciclos en búsqueda clásica si la mejora es marginal. |
| Clústeres de usuarios y películas son interpretables sobre los factores de SVD. | Usar los clústeres como **features contextuales** del modelo DL. |
| El split temporal expone explícitamente el cold-start de ítems. | Evaluar en la siguiente fase estrategias híbridas o content-based para cubrir ese hueco. |

### Siguientes notebooks
- `04_DeepLearning_Embeddings.ipynb` — NCF / Two-Tower con embeddings + los clústeres de este notebook como features categóricas.
- `05_Semantic_Search_RAG.ipynb` — buscador semántico sobre `tags` + `genome-scores`, re-ranker sobre los candidatos del SVD.


In [ ]:
# ============== PERSISTENCIA DE ARTEFACTOS ==============
with open(MODELS_DIR / 'svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
with open(MODELS_DIR / 'knn_model.pkl', 'wb') as f:
    pickle.dump(knn, f)
with open(MODELS_DIR / 'nmf_model.pkl', 'wb') as f:
    pickle.dump(nmf, f)
with open(MODELS_DIR / 'baseline_scores.pkl', 'wb') as f:
    pickle.dump({
        'bayesian_score_by_movieId': score_dict,
        'global_mean_C': float(C),
        'm_prior': float(m),
    }, f)
with open(MODELS_DIR / 'automl_winner.pkl', 'wb') as f:
    pickle.dump({
        'model':   automl_model,
        'name':    best_name,
        'params':  best_params,
        'results': automl_results,
    }, f)

user_cluster_df.to_parquet(DATA_INT_DIR / 'user_clusters.parquet', index=False)
item_cluster_df.to_parquet(DATA_INT_DIR / 'item_clusters.parquet', index=False)
comp_df.to_csv(DATA_INT_DIR / 'model_comparison.csv', index=False)

print('Artefactos persistidos:')
for p in sorted(MODELS_DIR.glob('*.pkl')):
    print(f'   {p.relative_to(ROOT)}   ({p.stat().st_size/1024:.1f} KB)')
for name in ('model_comparison.csv', 'user_clusters.parquet', 'item_clusters.parquet'):
    p = DATA_INT_DIR / name
    if p.is_file():
        print(f'   {p.relative_to(ROOT)}   ({p.stat().st_size/1024:.1f} KB)')

gc.collect()
print()
print('Fase 4-5 completada.')
